In [9]:
! pip install opencv-python


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:

import tkinter as tk
from tkinter import messagebox
import cv2
from PIL import Image, ImageTk


# ==========================================
# QR CODE CAMERA READER
# ==========================================

class QRCodeReaderApp:

    def __init__(self, root):

        self.root = root
        self.root.title("QR Code Camera Reader")
        self.root.geometry("1100x900")

        # Camera variables
        self.camera = None
        self.running = False

        # QR Code detector
        self.qr_detector = cv2.QRCodeDetector()

        # Keep reference to camera image
        self.photo = None

        # ==================================
        # TITLE
        # ==================================

        title = tk.Label(
            root,
            text="QR Code Camera Reader",
            font=("Arial", 22, "bold")
        )
        title.pack(pady=15)

        # ==================================
        # CAMERA DISPLAY
        # ==================================

        self.camera_label = tk.Label(
            root,
            text="Camera is not running",
            bg="black",
            fg="white",
            # width=60,
            # height=20
        )

        self.camera_label.pack(pady=10)

        # ==================================
        # START CAMERA BUTTON
        # ==================================

        self.start_button = tk.Button(
            root,
            text="Start Camera",
            font=("Arial", 12, "bold"),
            width=20,
            command=self.start_camera
        )

        self.start_button.pack(pady=5)

        # ==================================
        # STOP CAMERA BUTTON
        # ==================================

        self.stop_button = tk.Button(
            root,
            text="Stop Camera",
            font=("Arial", 12, "bold"),
            width=20,
            command=self.stop_camera
        )

        self.stop_button.pack(pady=5)

        # ==================================
        # QR CODE RESULT
        # ==================================

        result_label = tk.Label(
            root,
            text="QR Code Result:",
            font=("Arial", 12, "bold")
        )

        result_label.pack(pady=10)

        # Text box to display decoded result
        self.result_text = tk.Text(
            root,
            height=4,
            width=65,
            font=("Arial", 11)
        )

        self.result_text.pack(pady=5)

        # ==================================
        # COPY BUTTON
        # ==================================

        copy_button = tk.Button(
            root,
            text="Copy Result",
            width=20,
            command=self.copy_result
        )

        copy_button.pack(pady=5)

        # ==================================
        # STATUS
        # ==================================

        self.status_label = tk.Label(
            root,
            text="Ready",
            font=("Arial", 10)
        )

        self.status_label.pack(pady=10)

        # Handle closing window
        self.root.protocol(
            "WM_DELETE_WINDOW",
            self.close_application
        )

    # ==========================================
    # START CAMERA
    # ==========================================

    def start_camera(self):

        if self.running:
            return

        # Open default webcam
        self.camera = cv2.VideoCapture(0)

        # Check if camera opened successfully
        if not self.camera.isOpened():

            self.camera.release()
            self.camera = None

            messagebox.showerror(
                "Camera Error",
                "Could not open the camera."
            )

            return

        self.running = True

        self.status_label.config(
            text="Camera is running. Scan a QR code."
        )

        self.update_camera()

    # ==========================================
    # UPDATE CAMERA FRAME
    # ==========================================

    def update_camera(self):

        if not self.running:
            return

        # Read frame from camera
        success, frame = self.camera.read()

        if not success:

            self.status_label.config(
                text="Unable to read camera frame."
            )

            self.stop_camera()
            return

        # ==================================
        # DETECT QR CODE
        # ==================================

        decoded_text, points, _ = self.qr_detector.detectAndDecode(frame)

        # If QR code is detected
        if decoded_text:

            self.result_text.delete("1.0", tk.END)

            self.result_text.insert(
                tk.END,
                decoded_text
            )

            self.status_label.config(
                text="QR Code detected successfully!"
            )

        # ==================================
        # DRAW QR CODE BOUNDARY
        # ==================================

        if points is not None:

            points = points.astype(int)

            for i in range(len(points[0])):

                point1 = tuple(points[0][i])

                point2 = tuple(
                    points[0][(i + 1) % len(points[0])]
                )

                cv2.line(
                    frame,
                    point1,
                    point2,
                    (0, 255, 0),
                    3
                )

        # ==================================
        # CONVERT FRAME FOR TKINTER
        # ==================================

        # OpenCV uses BGR
        # Tkinter/PIL uses RGB
        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # Convert frame to PIL image
        image = Image.fromarray(frame_rgb)

        # Resize camera image
        image = image.resize(
            (640, 480)
        )

        # Convert to Tkinter image
        self.photo = ImageTk.PhotoImage(image)

        # Display image
        self.camera_label.config(
            image=self.photo,
            text=""
        )

        # Call update_camera again after 10 milliseconds
        self.root.after(
            10,
            self.update_camera
        )

    # ==========================================
    # STOP CAMERA
    # ==========================================

    def stop_camera(self):

        self.running = False

        if self.camera is not None:

            self.camera.release()
            self.camera = None

        self.camera_label.config(
            image="",
            text="Camera is not running",
            bg="black",
            fg="white"
        )

        self.photo = None

        self.status_label.config(
            text="Camera stopped."
        )

    # ==========================================
    # COPY QR CODE RESULT
    # ==========================================

    def copy_result(self):

        text = self.result_text.get(
            "1.0",
            tk.END
        ).strip()

        if text == "":

            messagebox.showwarning(
                "Empty Result",
                "No QR Code result to copy."
            )

            return

        self.root.clipboard_clear()

        self.root.clipboard_append(text)

        messagebox.showinfo(
            "Copied",
            "QR Code result copied to clipboard."
        )

    # ==========================================
    # CLOSE APPLICATION
    # ==========================================

    def close_application(self):

        self.stop_camera()

        self.root.destroy()


# ==========================================
# MAIN PROGRAM
# ==========================================

if __name__ == "__main__":

    root = tk.Tk()

    app = QRCodeReaderApp(root)

    root.mainloop()
